In [ ]:
import os
import re
import json
import time
import logging
from datasets import load_dataset, Dataset, DatasetDict
from translatepy.translators import YandexTranslate
from tqdm.auto import tqdm

# -------------------- Настройки --------------------
yandex = YandexTranslate()
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [ ]:
# Тест переводчика
test_text = "Hello, world!"
result = yandex.translate(test_text, "ru")
print(f"Original: {test_text}")
print(f"Translated: {result}")

In [33]:

SOURCE_REPO_ID = "DeepPavlov/wizard_of_wikipedia"
LOCAL_SAVE_PATH = "./wizard_of_wikipedia_ru"
CACHE_FILE = "translation_cache_wow.jsonl"
CONFIGS = ['corpus', 'queries']          # переводим только их
SPLITS = ['train', 'valid', 'test']

In [ ]:
# -------------------- Кэш --------------------
def load_cache():
    cache = {}
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    cache[obj["text"]] = obj["translation"]
                except:
                    continue
    return cache

def append_cache(text, translation):
    with open(CACHE_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps({"text": text, "translation": translation}, ensure_ascii=False) + "\n")

translation_cache = load_cache()
print(f"Записей в кэше: {len(translation_cache)}")

In [37]:
# -------------------- Улучшенный переводчик --------------------
def is_numeric_string(s: str) -> bool:
    """True, если в строке нет ни одной буквы."""
    return not bool(re.search(r'[A-Za-zА-Яа-яёЁ]', s))

def translate_text_robust(text, retries=3, delay=3):
    if not isinstance(text, str) or text.strip() == "":
        return "", True
    if is_numeric_string(text):
        return text, True
    if text in translation_cache:
        return translation_cache[text], True

    # Очень длинный (>20 000) → абзацы
    if len(text) > 20000:
        logging.info(f"Extremely long ({len(text)} chars), paragraphs...")
        paragraphs = re.split(r'\n\s*\n', text)
        if len(paragraphs) > 1:
            parts, ok = [], True
            for p in paragraphs:
                if not p.strip():
                    parts.append(p)
                    continue
                t, o = translate_text_robust(p, retries, delay)
                if not o: ok = False; break
                parts.append(t)
            if ok:
                full = '\n\n'.join(parts)
                translation_cache[text] = full
                append_cache(text, full)
                return full, True

    # Длинный (>8 000) → предложения
    if len(text) > 8000:
        logging.info(f"Long ({len(text)} chars), sentences...")
        sentences = re.split(r'(?<=[.!?])\s+', text)
        if len(sentences) > 1:
            parts, ok = [], True
            for s in sentences:
                t, o = translate_text_robust(s, retries, delay)
                if not o: ok = False; break
                parts.append(t)
            if ok:
                full = ' '.join(parts)
                translation_cache[text] = full
                append_cache(text, full)
                return full, True

    last_exception = None
    for attempt in range(retries):
        try:
            time.sleep(0.5 if attempt == 0 else delay)
            result = yandex.translate(text, "ru")
            translated = str(result.result) if hasattr(result, 'result') else str(result)
            translation_cache[text] = translated
            append_cache(text, translated)
            return translated, True
        except Exception as e:
            last_exception = e
            error_str = str(e).lower()
            if '413' in error_str or 'too long' in error_str:
                logging.info(f"413 error, splitting: '{text[:30]}...'")
                sentences = re.split(r'(?<=[.!?])\s+', text)
                if len(sentences) > 1:
                    parts, ok = [], True
                    for s in sentences:
                        t, o = translate_text_robust(s, 1, delay)
                        if not o: ok = False; break
                        parts.append(t)
                    if ok:
                        full = ' '.join(parts)
                        translation_cache[text] = full
                        append_cache(text, full)
                        return full, True
            if any(c in error_str for c in ['502','503','504']):
                time.sleep(delay * (attempt+1))
            elif '429' in error_str:
                time.sleep(delay*4 + 10)
            else:
                time.sleep(delay)

    # Последний шанс
    if len(text) > 2000:
        logging.info("Final split attempt...")
        sentences = re.split(r'(?<=[.!?])\s+', text)
        if len(sentences) > 1:
            parts, ok = [], True
            for s in sentences:
                t, o = translate_text_robust(s, 1, delay)
                if not o: ok = False; break
                parts.append(t)
            if ok:
                full = ' '.join(parts)
                translation_cache[text] = full
                append_cache(text, full)
                return full, True

    logging.error(f"Failed: '{text[:50]}...' Error: {last_exception}")
    return "", False

# -------------------- Перевод конкретных полей --------------------
def translate_fields_in_example(example, fields):
    """Переводит указанные поля. Если поле — список сообщений (как 'text' в queries),
    переводим только содержимое ключа 'content' внутри каждого сообщения."""
    result = dict(example)
    all_ok = True

    for field in fields:
        value = example.get(field)

        if isinstance(value, str):
            trans, ok = translate_text_robust(value)
            if not ok:
                all_ok = False
            result[field + '_ru'] = trans if ok else value

        elif isinstance(value, list) and value and isinstance(value[0], dict):
            # Список сообщений (пример: [{'content': '...', 'role': '...'}, ...])
            trans_list = []
            for msg in value:
                new_msg = dict(msg)                     # копируем все поля (role, ...)
                if 'content' in msg and isinstance(msg['content'], str):
                    cont, ok = translate_text_robust(msg['content'])
                    if not ok:
                        all_ok = False
                    new_msg['content'] = cont if ok else msg['content']
                trans_list.append(new_msg)
            result[field + '_ru'] = trans_list

        elif isinstance(value, list):
            # Обычный список строк
            trans_list = []
            for item in value:
                if isinstance(item, str):
                    t, ok = translate_text_robust(item)
                    if not ok:
                        all_ok = False
                    trans_list.append(t if ok else item)
                else:
                    trans_list.append(item)
            result[field + '_ru'] = trans_list
        else:
            # Числа, None и т.п. – не переводим
            result[field + '_ru'] = value

    result['_success'] = all_ok
    return result
    
# -------------------- Обработка сплита --------------------
def process_config_split(config_name, split_name, source_split, fields, progress_file):
    translated_records = []
    failed_indices = set()

    if os.path.exists(progress_file):
        with open(progress_file, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    if rec.get('_failed', False):
                        failed_indices.add(rec['_index'])
                    else:
                        translated_records.append(rec)
                except:
                    continue
        logging.info(f"[{config_name}/{split_name}] Resuming: {len(translated_records)} ok, {len(failed_indices)} failed")

    start_index = len(translated_records) + len(failed_indices)
    total = len(source_split)

    if start_index < total:
        logging.info(f"[{config_name}/{split_name}] Starting from index {start_index}...")
        with open(progress_file, "a", encoding="utf-8") as f:
            pbar = tqdm(
                enumerate(source_split.select(range(start_index, total))),
                desc=f"Translating {config_name}/{split_name}",
                total=total - start_index
            )
            for idx, example in pbar:
                global_idx = start_index + idx
                if global_idx in {r.get('_index', -1) for r in translated_records}:
                    continue

                translated = translate_fields_in_example(example, fields)
                record_out = {
                    '_index': global_idx,
                    '_failed': not translated['_success'],
                    **translated
                }
                del record_out['_success']

                f.write(json.dumps(record_out, ensure_ascii=False) + "\n")
                f.flush()
                time.sleep(0.5)

                if not record_out['_failed']:
                    translated_records.append(record_out)
                else:
                    failed_indices.add(global_idx)

    all_successful = []
    if os.path.exists(progress_file):
        with open(progress_file, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    if not rec.get('_failed', False):
                        rec.pop('_index', None)
                        rec.pop('_failed', None)
                        all_successful.append(rec)
                except:
                    continue

    if not all_successful:
        logging.error(f"[{config_name}/{split_name}] No successful records!")
        return None

    ok_cnt = len(all_successful)
    fail_cnt = total - ok_cnt
    logging.info(f"[{config_name}/{split_name}] Done: {ok_cnt}/{total} ({fail_cnt} failed)")
    return Dataset.from_list(all_successful)

In [ ]:
# -------------------- Запуск --------------------
logging.info("Loading dataset...")
all_configs = {}
for config in CONFIGS:
    source = load_dataset(SOURCE_REPO_ID, config)
    config_splits = {}
    # Определяем поля для перевода
    if config == 'corpus':
        fields = ['title', 'text']
    elif config == 'queries':
        fields = ['topic', 'text', 'persona']
    else:
        fields = []

    for split in SPLITS:
        if split not in source:
            logging.warning(f"Split '{split}' not found in {config}, skipping")
            continue
        progress_file = f"translated_wow_{config}_{split}.jsonl"
        ds = process_config_split(config, split, source[split], fields, progress_file)
        if ds is not None:
            config_splits[split] = ds
    if config_splits:
        all_configs[config] = DatasetDict(config_splits)

In [ ]:
# Добавляем qrels как есть
try:
    qrels = load_dataset(SOURCE_REPO_ID, 'qrels')
    all_configs['qrels'] = qrels
    print("qrels добавлены без перевода")
except Exception as e:
    logging.warning(f"Не удалось загрузить qrels: {e}")

if all_configs:
    final_dataset = DatasetDict(all_configs)
    final_dataset.save_to_disk(LOCAL_SAVE_PATH)
    print(f"\nСохранено в {LOCAL_SAVE_PATH}")

    # Сравнение с оригиналом
    print("\n" + "="*60)
    print("СРАВНЕНИЕ С ОРИГИНАЛОМ")
    print("="*60)
    for config in all_configs:
        orig = load_dataset(SOURCE_REPO_ID, config)
        for split in orig:
            our_len = len(all_configs[config][split])
            orig_len = len(orig[split])
            status = "✅" if our_len == orig_len else "❌"
            print(f"{status} {config}/{split}: orig={orig_len}, ours={our_len}")
else:
    print("Не удалось перевести ни одной конфигурации")

In [ ]:
import os, json
from datasets import Dataset, DatasetDict, load_dataset
from huggingface_hub import login

login(token="YOUR_HF_TOKEN")   # замените на ваш токен

REPO_ID = "DeepPavlov/wizard_of_wikipedia_ru"
SPLITS = ['train', 'valid', 'test']   # обратите внимание: valid, не validation



In [ ]:
# Загрузка на Hugging Face Hub
for config_name, ds_dict in final_dataset.items():
    ds_dict.push_to_hub(
        REPO_ID,
        config_name=config_name,
        private=False,
        commit_message="Full Russian translation – corpus & queries translated, qrels original"
    )
    print(f"{config_name} uploaded")

print(f"Готово: https://huggingface.co/datasets/{REPO_ID}")